# 02 Silver Transformations
Cleans Bronze data and creates curated Silver tables.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

dbutils.widgets.text("catalog", "health_cat", "Catalog")
dbutils.widgets.text("bronze_schema", "bronze", "Bronze schema")
dbutils.widgets.text("silver_schema", "silver", "Silver schema")
dbutils.widgets.text("write_mode", "overwrite", "Write mode")

catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
silver_schema = dbutils.widgets.get("silver_schema")
write_mode = dbutils.widgets.get("write_mode")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{silver_schema}")

def write_delta_table(df, table_name):
    writer = df.write.mode(write_mode).format("delta")
    if write_mode == "overwrite":
        writer = writer.option("overwriteSchema", "true")
    writer.saveAsTable(f"{catalog}.{silver_schema}.{table_name}")

In [0]:
appointments = spark.table(f"{catalog}.{bronze_schema}.appointments")
doctors = spark.table(f"{catalog}.{bronze_schema}.doctors")
departments = spark.table(f"{catalog}.{bronze_schema}.departments")
billing = spark.table(f"{catalog}.{bronze_schema}.billing")
patients = spark.table(f"{catalog}.{bronze_schema}.patients")

def standardize_text(column):
    return F.initcap(F.regexp_replace(F.trim(column), r"\s+", " "))

doctors_dim = doctors.select(
    F.col("doctor_id").cast("int").alias("doctor_id"),
    standardize_text(F.col("doctor_name")).alias("doctor_name"),
    F.col("department_id").cast("int").alias("department_id"),
    standardize_text(F.col("specialization")).alias("specialization"),
    F.col("status").alias("doctor_status"),
)
departments_dim = departments.select(
    F.col("department_id").cast("int").alias("department_id"),
    standardize_text(F.col("department_name")).alias("department_name"),
    standardize_text(F.col("location")).alias("location"),
)
billing_fact = billing.select(
    F.col("appointment_id").cast("int").alias("appointment_id"),
    F.col("bill_amount").cast("decimal(10,2)").alias("bill_amount"),
    standardize_text(F.col("payment_status")).alias("payment_status"),
    standardize_text(F.col("payment_method")).alias("payment_method"),
    F.to_timestamp("billing_datetime").alias("billing_datetime"),
)
patients_dim = patients.select(
    F.col("patient_id").cast("int").alias("patient_id"),
    standardize_text(F.col("patient_name")).alias("patient_name"),
    F.upper(F.trim("gender")).alias("gender"),
    F.to_date("date_of_birth").alias("date_of_birth"),
    standardize_text(F.col("city")).alias("city"),
    standardize_text(F.col("state")).alias("state"),
).withColumn(
    "patient_age",
    F.floor(F.months_between(F.current_date(), F.col("date_of_birth")) / 12).cast("int"),
)

write_delta_table(doctors_dim, "doctors_clean")
write_delta_table(departments_dim, "departments_clean")
write_delta_table(patients_dim, "patients_clean")

silver_appointments = (
    appointments
    .dropDuplicates(["appointment_id"])
    .withColumn("appointment_id", F.col("appointment_id").cast("int"))
    .withColumn("patient_id", F.col("patient_id").cast("int"))
    .withColumn("doctor_id", F.col("doctor_id").cast("int"))
    .withColumn(
        "no_show_int",
        F.when(F.lower(F.col("no_show_flag").cast("string")).isin("true", "t", "1", "yes", "y"), 1).otherwise(0),
    )
    .withColumn("no_show_flag_clean", F.col("no_show_int") == 1)
    .withColumn(
        "appointment_status_clean",
        F.when(F.lower(F.trim("appointment_status")).isin("completed", "complete"), "Completed")
        .when(F.lower(F.trim("appointment_status")).isin("no show", "no-show", "noshow"), "No Show")
        .when(F.lower(F.trim("appointment_status")).isin("cancelled", "canceled"), "Cancelled")
        .otherwise(standardize_text(F.col("appointment_status"))),
    )
    .withColumn("appointment_ts", F.to_timestamp("appointment_datetime"))
    .withColumn("checkin_ts", F.to_timestamp("checkin_datetime"))
    .withColumn("consultation_start_ts", F.to_timestamp("consultation_start_datetime"))
    .withColumn("consultation_end_ts", F.to_timestamp("consultation_end_datetime"))
    .withColumn("visit_month", F.date_format("appointment_ts", "yyyy-MM"))
    .withColumn("wait_time_minutes", (F.col("consultation_start_ts").cast("long") - F.col("checkin_ts").cast("long")) / 60)
    .withColumn("has_null_patient_id", F.col("patient_id").isNull())
    .withColumn("has_null_doctor_id", F.col("doctor_id").isNull())
    .withColumn("has_invalid_appointment_ts", F.col("appointment_ts").isNull())
)

write_delta_table(silver_appointments, "appointments_clean")

In [0]:
appointments_enriched = (
    silver_appointments.alias("a")
    .join(doctors_dim.alias("d"), "doctor_id", "left")
    .join(departments_dim.alias("dept"), "department_id", "left")
    .join(billing_fact.alias("b"), "appointment_id", "left")
    .join(patients_dim.alias("p"), "patient_id", "left")
    .withColumn("is_unmatched_doctor", F.col("doctor_name").isNull())
    .withColumn("is_unmatched_patient", F.col("patient_name").isNull())
    .withColumn("is_valid_record", ~(F.col("has_null_patient_id") | F.col("has_null_doctor_id") | F.col("has_invalid_appointment_ts") | F.col("is_unmatched_doctor") | F.col("is_unmatched_patient")))
)

write_delta_table(appointments_enriched, "appointments_enriched")

display(appointments_enriched)

patient_id,appointment_id,department_id,doctor_id,appointment_datetime,checkin_datetime,consultation_start_datetime,consultation_end_datetime,appointment_status,no_show_flag,reason_for_visit,created_at,updated_at,load_time,batch_id,source_system,source_file,no_show_int,no_show_flag_clean,appointment_status_clean,appointment_ts,checkin_ts,consultation_start_ts,consultation_end_ts,visit_month,wait_time_minutes,has_null_patient_id,has_null_doctor_id,has_invalid_appointment_ts,doctor_name,specialization,doctor_status,department_name,location,bill_amount,payment_status,payment_method,billing_datetime,patient_name,gender,date_of_birth,city,state,patient_age,is_unmatched_doctor,is_unmatched_patient,is_valid_record
5,5,4,5,2026-04-03T09:30:00.000Z,null,null,null,Cancelled,f,Child fever consultation,2026-04-21T23:32:36.867Z,2026-04-21T23:32:36.867Z,2026-04-21T19:27:37.731Z,20260421192700,postgresql,s3://healthcare-ops-capstone-siddh-20260421235653/raw/postgresql/appointments.csv,0,false,Cancelled,2026-04-03T09:30:00.000Z,null,null,null,2026-04,null,false,false,false,Dr. Neha Iyer,Child Health,Active,Pediatrics,Block A - Floor 3,null,null,null,null,Imran Khan,MALE,1969-11-05,Mumbai,Maharashtra,56,false,false,true
1,1,1,1,2026-04-01T09:00:00.000Z,2026-04-01T08:55:00.000Z,2026-04-01T09:10:00.000Z,2026-04-01T09:35:00.000Z,Completed,f,Chest pain,2026-04-21T23:32:36.867Z,2026-04-21T23:32:36.867Z,2026-04-21T19:27:37.731Z,20260421192700,postgresql,s3://healthcare-ops-capstone-siddh-20260421235653/raw/postgresql/appointments.csv,0,false,Completed,2026-04-01T09:00:00.000Z,2026-04-01T08:55:00.000Z,2026-04-01T09:10:00.000Z,2026-04-01T09:35:00.000Z,2026-04,15.0,false,false,false,Dr. Ananya Sharma,Interventional Cardiology,Active,Cardiology,Block A - Floor 2,1500.00,Paid,Upi,2026-04-01T09:40:00.000Z,Amit Kumar,MALE,1985-04-12,Hyderabad,Telangana,41,false,false,true
3,3,2,3,2026-04-02T11:00:00.000Z,2026-04-02T10:50:00.000Z,2026-04-02T11:20:00.000Z,2026-04-02T11:45:00.000Z,Completed,f,Knee pain,2026-04-21T23:32:36.867Z,2026-04-21T23:32:36.867Z,2026-04-21T19:27:37.731Z,20260421192700,postgresql,s3://healthcare-ops-capstone-siddh-20260421235653/raw/postgresql/appointments.csv,0,false,Completed,2026-04-02T11:00:00.000Z,2026-04-02T10:50:00.000Z,2026-04-02T11:20:00.000Z,2026-04-02T11:45:00.000Z,2026-04,30.0,false,false,false,Dr. Priya Nair,Joint Replacement,Active,Orthopedics,Block B - Floor 1,2200.00,Paid,Credit Card,2026-04-02T11:50:00.000Z,Rahul Verma,MALE,1978-01-30,Bengaluru,Karnataka,48,false,false,true
2,2,1,2,2026-04-01T10:00:00.000Z,null,null,null,No Show,t,Follow-up consultation,2026-04-21T23:32:36.867Z,2026-04-21T23:32:36.867Z,2026-04-21T19:27:37.731Z,20260421192700,postgresql,s3://healthcare-ops-capstone-siddh-20260421235653/raw/postgresql/appointments.csv,1,true,No Show,2026-04-01T10:00:00.000Z,null,null,null,2026-04,null,false,false,false,Dr. Rohan Mehta,General Cardiology,Active,Cardiology,Block A - Floor 2,null,null,null,null,Sneha Reddy,FEMALE,1992-09-21,Hyderabad,Telangana,33,false,false,true
6,6,4,5,2026-04-03T10:30:00.000Z,2026-04-03T10:20:00.000Z,2026-04-03T10:45:00.000Z,2026-04-03T11:05:00.000Z,Completed,f,Routine child checkup,2026-04-21T23:32:36.867Z,2026-04-21T23:32:36.867Z,2026-04-21T19:27:37.731Z,20260421192700,postgresql,s3://healthcare-ops-capstone-siddh-20260421235653/raw/postgresql/appointments.csv,0,false,Completed,2026-04-03T10:30:00.000Z,2026-04-03T10:20:00.000Z,2026-04-03T10:45:00.000Z,2026-04-03T11:05:00.000Z,2026-04,25.0,false,false,false,Dr. Neha Iyer,Child Health,Active,Pediatrics,Block A - Floor 3,1200.00,Pending,Insurance,2026-04-03T11:10:00.000Z,Kavya Menon,FEMALE,2015-03-25,Kochi,Kerala,11,false,false,true
7,7,5,6,2026-04-04T14:00:00.000Z,null,null,null,No Show,t,Headache and dizziness,2026-04-21T23:32:36.867Z,2026-04-21T23:32:36.867Z,2026-04-21T19:27:37.731Z,20260421192700,postgresql,s3://healthcare-ops-capstone-siddh-20260421235653/raw/postgresql/appointments.csv,1,true,No Show,2026-04-04T14:00:00.000Z,null,null

In [0]:
invalid_appointments = appointments_enriched.filter(~F.col("is_valid_record")).select(
    "appointment_id",
    "patient_id",
    "doctor_id",
    "appointment_status",
    "appointment_status_clean",
    "has_null_patient_id",
    "has_null_doctor_id",
    "has_invalid_appointment_ts",
    "is_unmatched_doctor",
    "is_unmatched_patient",
    "source_file",
    "load_time",
    "batch_id",
)

write_delta_table(invalid_appointments, "dq_invalid_appointments")

dq_summary = spark.createDataFrame(
    [
        ("appointments_total", appointments_enriched.count()),
        ("appointments_invalid", invalid_appointments.count()),
        ("null_patient_id", appointments_enriched.filter(F.col("has_null_patient_id")).count()),
        ("null_doctor_id", appointments_enriched.filter(F.col("has_null_doctor_id")).count()),
        ("invalid_appointment_ts", appointments_enriched.filter(F.col("has_invalid_appointment_ts")).count()),
        ("unmatched_doctor", appointments_enriched.filter(F.col("is_unmatched_doctor")).count()),
        ("unmatched_patient", appointments_enriched.filter(F.col("is_unmatched_patient")).count()),
    ],
    ["dq_rule", "record_count"],
)

write_delta_table(dq_summary, "dq_appointments_summary")
display(dq_summary)

dq_rule,record_count
appointments_total,8
appointments_invalid,0
null_patient_id,0
null_doctor_id,0
invalid_appointment_ts,0
unmatched_doctor,0
unmatched_patient,0


In [0]:
kaggle = spark.table(f"{catalog}.{bronze_schema}.kaggle_noshow_appointments")
kaggle_clean = (
    kaggle
    .withColumnRenamed("PatientId", "patient_id")
    .withColumnRenamed("AppointmentID", "appointment_id")
    .withColumnRenamed("ScheduledDay", "scheduled_day")
    .withColumnRenamed("AppointmentDay", "appointment_day")
    .withColumnRenamed("No-show", "no_show")
    .withColumn("scheduled_ts", F.to_timestamp("scheduled_day"))
    .withColumn("appointment_date", F.to_date("appointment_day"))
    .withColumn("no_show_int", F.when(F.lower(F.col("no_show")) == "yes", 1).otherwise(0))
    .withColumn("no_show_flag_clean", F.col("no_show_int") == 1)
    .withColumn("lead_days", F.datediff("appointment_date", F.to_date("scheduled_ts")))
)

write_delta_table(kaggle_clean, "kaggle_noshow_clean")
display(kaggle_clean.limit(20))


patient_id,appointment_id,Gender,scheduled_day,appointment_day,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,no_show,load_time,batch_id,source_system,source_file,scheduled_ts,appointment_date,no_show_int,no_show_flag_clean,lead_days
2.9872499824296E13,5642903,F,2016-04-29T18:38:08.000Z,2016-04-29T00:00:00.000Z,62,JARDIM DA PENHA,0,1,0,0,0,0,No,2026-04-21T19:28:26.222Z,20260421192700,kaggle,s3://healthcare-ops-capstone-siddh-20260421235653/raw/external/kaggle_noshow_appointments/kaggle_noshow_appointments.csv,2016-04-29T18:38:08.000Z,2016-04-29,0,false,0
5.58997776694438E14,5642503,M,2016-04-29T16:08:27.000Z,2016-04-29T00:00:00.000Z,56,JARDIM DA PENHA,0,0,0,0,0,0,No,2026-04-21T19:28:26.222Z,20260421192700,kaggle,s3://healthcare-ops-capstone-siddh-20260421235653/raw/external/kaggle_noshow_appointments/kaggle_noshow_appointments.csv,2016-04-29T16:08:27.000Z,2016-04-29,0,false,0
4.262962299951E12,5642549,F,2016-04-29T16:19:04.000Z,2016-04-29T00:00:00.000Z,62,MATA DA PRAIA,0,0,0,0,0,0,No,2026-04-21T19:28:26.222Z,20260421192700,kaggle,s3://healthcare-ops-capstone-siddh-20260421235653/raw/external/kaggle_noshow_appointments/kaggle_noshow_appointments.csv,2016-04-29T16:19:04.000Z,2016-04-29,0,false,0
8.67951213174E11,5642828,F,2016-04-29T17:29:31.000Z,2016-04-29T00:00:00.000Z,8,PONTAL DE CAMBURI,0,0,0,0,0,0,No,2026-04-21T19:28:26.222Z,20260421192700,kaggle,s3://healthcare-ops-capstone-siddh-20260421235653/raw/external/kaggle_noshow_appointments/kaggle_noshow_appointments.csv,2016-04-29T17:29:31.000Z,2016-04-29,0,false,0
8.841186448183E12,5642494,F,2016-04-29T16:07:23.000Z,2016-04-29T00:00:00.000Z,56,JARDIM DA PENHA,0,1,1,0,0,0,No,2026-04-21T19:28:26.222Z,20260421192700,kaggle,s3://healthcare-ops-capstone-siddh-20260421235653/raw/external/kaggle_noshow_appointments/kaggle_noshow_appointments.csv,2016-04-29T16:07:23.000Z,2016-04-29,0,false,0
9.5985133231274E13,5626772,F,2016-04-27T08:36:51.000Z,2016-04-29T00:00:00.000Z,76,REPÚBLICA,0,1,0,0,0,0,No,2026-04-21T19:28:26.222Z,20260421192700,kaggle,s3://healthcare-ops-capstone-siddh-20260421235653/raw/external/kaggle_noshow_appointments/kaggle_noshow_appointments.csv,2016-04-27T08:36:51.000Z,2016-04-29,0,false,2
7.33688164476661E14,5630279,F,2016-04-27T15:05:12.000Z,2016-04-29T00:00:00.000Z,23,GOIABEIRAS,0,0,0,0,0,0,Yes,2026-04-21T19:28:26.222Z,20260421192700,kaggle,s3://healthcare-ops-capstone-siddh-20260421235653/raw/external/kaggle_noshow_appointments/kaggle_noshow_appointments.csv,2016-04-27T15:05:12.000Z,2016-04-29,1,true,2
3.449833394123E12,5630575,F,2016-04-27T15:39:58.000Z,2016-04-29T00:00:00.000Z,39,GOIABEIRAS,0,0,0,0,0,0,Yes,2026-04-21T19:28:26.222Z,20260421192700,kaggle,s3://healthcare-ops-capstone-siddh-20260421235653/raw/external/kaggle_noshow_appointments/kaggle_noshow_appointments.csv,2016-04-27T15:39:58.000Z,2016-04-29,1,true,2
5.6394729949972E13,5638447,F,2016-04-29T08:02:16.000Z,2016-04-29T00:00:00.000Z,21,ANDORINHAS,0,0,0,0,0,0,No,2026-04-21T19:28:26.222Z,20260421192700,kaggle,s3://healthcare-ops-capstone-siddh-20260421235653/raw/external/kaggle_noshow_appointments/kaggle_noshow_appointments.csv,2016-04-29T08:02:16.000Z,2016-04-29,0,false,0
7.8124564369297E13,5629123,F,2016-04-27T12:48:25.000Z,2016-04-29T00:00:00.000Z,19,CONQUISTA,0,0,0,0,0,0,No,2026-04-21T19:28:26.222Z,20260421192700,kaggle,s3://healthcare-ops-capstone-siddh-20260421235653/raw/external/kaggle_noshow_appointments/kaggle_noshow_appointments.csv,2016-04-27T12:48:25.000Z,2016-04-29,0,false,2
